# 00 — Methodology checks (XAI/CMI parameter justification)

The XAI/CMI phase has parameters that are **justified empirically, not derived** — the first ladder
parameters in this thesis chosen from measured evidence rather than from signal theory or convention. This
notebook is the executable, re-runnable record of how they were decided. It is **model-agnostic** (loads
checkpoints from `sleep_edf/results/checkpoints/`); it is not part of any single rung.

Checks:
- **§1** KernelSHAP convergence vs `n_samples` (Model 2) — sets `n_samples = 8000`.
- **§2** the same on Model 5 — tests that the residual under-sampling bias is ~uniform across the ladder.
- **§3** CMI/PES stability vs evaluation-sample count `N` — was the §1 CMI movement partly noise, and is
  `N = 500` enough?
- **§4** class-tracking (predicted vs true class) — sets the ladder-wide target-class rule.
- **§5** perturbation method — the `zero` vs `laplace` decision.
- **§6** device selection — the per-method CPU/MPS split and its dependence on coalition batching.

> **You run this.** §1–§3 are compute-heavy (minutes each; per-cell estimates are noted). §4–§6 are light.
> Prepared unexecuted.

## §0. Setup — imports, config, shared helpers

In [ ]:
import sys, json, time
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "sleep_edf").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from sleep_edf.loader import load_sleep_edf
import sleep_edf.config as cfg
from harness.models.cnn import build_cnn, torch_predict_proba
from harness.models.transformer import build_transformer
from harness.xai.regions import build_region_grid
from harness.xai.kernel_shap import kernel_shap
from harness.xai.deletion_curves import perturbation_curves
from harness.xai.cmi import compute_cmi, decaying_degradation_score, pes as pes_of, CMI as CMI_of

CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"]
CKPT_DIR = PROJECT_ROOT / "sleep_edf" / "results" / "checkpoints"
FIG_DIR  = PROJECT_ROOT / "sleep_edf" / "results" / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

# The 50-region CMI grid (3000 samples, 2% -> 50 regions of 60), identical to the ladder XAI phase.
grid = build_region_grid(cfg.INPUT_LENGTH, cfg.REGION_SIZE_PRIMARY_PCT)

# Device + batching policy for KernelSHAP (see §6): batching is a *large* win on MPS, a loss on CPU,
# so batch (perturbations_per_eval=200) only when on MPS; pe=1 on CPU.
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
KS_PE  = 200 if DEVICE == "mps" else 1
SAMPLE_SEED = 42                          # fixed sample selection (shared across every check)
print(f"grid: {grid.n_regions} regions of {int(np.unique(grid.sizes)[0])} | device={DEVICE} | KS perturbations_per_eval={KS_PE}")

X_test, y_test = load_sleep_edf("test", verbose=False)

def stratified_idx(y, per_class, seed=SAMPLE_SEED):
    """Deterministic per-class stratified indices: `per_class` from each of the 5 classes."""
    rng = np.random.RandomState(seed); idx = []
    for c in range(cfg.N_CLASSES):
        idx += list(rng.choice(np.where(y == c)[0], per_class, replace=False))
    return np.array(idx)

def load_model2(seed=0, device=DEVICE):
    m = build_cnn("shallow", n_classes=cfg.N_CLASSES, in_channels=cfg.IN_CHANNELS, kernel_size=cfg.CNN_KERNEL_SIZE)
    m.load_state_dict(torch.load(CKPT_DIR / f"model2_shallow_cnn_seed{seed}.pt", map_location="cpu")); m.eval()
    return m.to(device)

def load_model5(seed=0, device=DEVICE):
    m = build_transformer(input_length=cfg.INPUT_LENGTH, in_channels=cfg.IN_CHANNELS,
                          n_classes=cfg.N_CLASSES, patch_size=cfg.TRANSFORMER_PATCH_SIZE)
    m.load_state_dict(torch.load(CKPT_DIR / f"model5_transformer_seed{seed}.pt", map_location="cpu")); m.eval()
    return m.to(device)

# The 10 stratified sweep samples (2 per class, seed 42) — the SAME set §1/§2/§4 use.
SWEEP_IDX  = stratified_idx(y_test, 2, SAMPLE_SEED)
SWEEP_SIGS = [X_test[i].astype(float) for i in SWEEP_IDX]
print("10 sweep samples per-class:", {CLASS_NAMES[c]: int((y_test[SWEEP_IDX]==c).sum()) for c in range(5)})

def ks_sweep(pp, sigs, ns_list, pe=KS_PE):
    """KernelSHAP attribution (zero PM, predicted class) for each n_samples in ns_list -> {n: (S,50)}."""
    attrs = {}
    for n in ns_list:
        t0 = time.perf_counter()
        attrs[n] = np.array([kernel_shap(pp, s, grid, "zero", n_samples=n, perturbations_per_eval=pe) for s in sigs])
        print(f"  n_samples={n:>5}: {(time.perf_counter()-t0)/len(sigs)*1000:6.0f} ms/sample", flush=True)
    return attrs

def cmi_of_attrs(pp, sigs, attr_mat):
    """CMI over the given samples for one attribution matrix (deletion curves, zero PM, predicted class)."""
    M, L = [], []
    for s, a in zip(sigs, attr_mat):
        c = perturbation_curves(pp, s, grid, a, method="zero")
        M.append(c["MoRF"]); L.append(c["LeRF"])
    return compute_cmi(M, L)

def consecutive_spearman(attrs, ns_list):
    """Mean & min Spearman between attributions at consecutive n_samples (per-sample, then aggregated)."""
    rows = []
    for a, b in zip(ns_list[:-1], ns_list[1:]):
        sp = [spearmanr(attrs[a][i], attrs[b][i]).correlation for i in range(len(attrs[a]))]
        rows.append({"pair": f"{a}->{b}", "mean_spearman": float(np.mean(sp)), "min_spearman": float(np.min(sp))})
    return rows

SWEEP_NS = [200, 500, 1000, 2000, 4000, 8000]
print("sweep n_samples:", SWEEP_NS)

# Model 2 seed 0 predict_proba — shared by §1, §3, §4 (so those sections don't depend on run order).
pp2 = torch_predict_proba(load_model2(seed=0), device=DEVICE)
print("pp2 ready (Model 2 seed 0 on", DEVICE + ")")

## §1. KernelSHAP convergence over the 50-region grid

**What this tests.** Whether KernelSHAP's 50-region attribution *settles* as `n_samples` grows (200 → 16000),
on the same 10 stratified samples, for Model 2 and Model 5 independently.

**What would count as convergence.** Two things together: consecutive-setting **Spearman → 1.0** (doubling
`n_samples` stops changing the ranking) **and CMI plateauing**. The Spearman is the load-bearing evidence — a
rising CMI alone does not show the *attributions* have stopped moving; the rank stability does.

**Per model** below: one table (`n_samples`, CMI, consecutive-setting Spearman, ms/sample), then a convergence
plot, then the conclusion. Read each model as its own convergence indicator — this is not a cross-model
comparison. **Preview:** neither model settles by n=16000, so `n=8000` was retained (raising it changes nothing
but roughly doubles cost).

In [ ]:
# §1 — sweep both models over n_samples 200->16000, capturing CMI, consecutive Spearman and per-sample timing.
NS_CONV = globals().get("NS_CONV", SWEEP_NS + [16000])          # 200..8000 (from §0) + 16000
def timed_ks_sweep(pp, sigs, ns_list, pe=KS_PE):
    attrs, ms = {}, {}
    for n in ns_list:
        t0 = time.perf_counter()
        attrs[n] = np.array([kernel_shap(pp, s, grid, "zero", n_samples=n, perturbations_per_eval=pe) for s in sigs])
        ms[n] = (time.perf_counter() - t0) / len(sigs) * 1000
    return attrs, ms
def convergence_table(pp, sigs, ns_list, label):
    attrs, ms = timed_ks_sweep(pp, sigs, ns_list)
    sp = {r["pair"]: r["mean_spearman"] for r in consecutive_spearman(attrs, ns_list)}
    cmi = {n: cmi_of_attrs(pp, sigs, attrs[n])["CMI"] for n in ns_list}
    print(f"\n{label} — KernelSHAP convergence (10 stratified samples)")
    print(f"{'n_samples':>9}{'CMI':>8}{'consec-Spearman':>17}{'ms/sample':>11}")
    for k, n in enumerate(ns_list):
        consec = "—" if k == 0 else f"{sp[f'{ns_list[k-1]}->{n}']:.3f}"
        print(f"{n:>9}{cmi[n]:>8.3f}{consec:>17}{ms[n]:>11.0f}")
    return {"attrs": attrs, "ms": ms, "sp": sp, "cmi": cmi, "ns": ns_list}

print(f"Model 2 (CNN) — sweeping n_samples {NS_CONV} on {DEVICE} (pe={KS_PE}) ...", flush=True)
m2_conv = convergence_table(pp2, SWEEP_SIGS, NS_CONV, "Model 2 (shallow CNN)")

In [ ]:
# §1 — Model 5 (transformer), same sweep and settings. pp5 defined here (only the convergence section needs it).
pp5 = torch_predict_proba(load_model5(seed=0), device=DEVICE)
print(f"Model 5 (transformer) — sweeping n_samples {NS_CONV} on {DEVICE} (pe={KS_PE}) ...", flush=True)
m5_conv = convergence_table(pp5, SWEEP_SIGS, NS_CONV, "Model 5 (transformer)")

# light reproduction anchor (per model, no cross-model gap): CMI at the top setting
print(f"\nreproduction: CMI@{NS_CONV[-1]}  Model 2 {m2_conv['cmi'][NS_CONV[-1]]:.3f} (≈0.401) | "
      f"Model 5 {m5_conv['cmi'][NS_CONV[-1]]:.3f} (≈0.665)")

In [ ]:
# §1 — convergence plot: CMI still rising + consecutive-setting Spearman still below 1.0. Per-model lines.
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for conv, mk, lab in [(m2_conv, "o-", "Model 2 (CNN)"), (m5_conv, "s-", "Model 5 (transformer)")]:
    ax[0].plot(conv["ns"], [conv["cmi"][n] for n in conv["ns"]], mk, label=lab)
ax[0].set_xscale("log"); ax[0].set_xlabel("n_samples"); ax[0].set_ylabel("CMI (10 samples)")
ax[0].set_title("CMI still rising at n=16000 → not plateaued"); ax[0].legend(fontsize=8)
for conv, mk, lab in [(m2_conv, "o-", "Model 2 (CNN)"), (m5_conv, "s-", "Model 5 (transformer)")]:
    pairs = [r["pair"] for r in consecutive_spearman(conv["attrs"], conv["ns"])]
    ax[1].plot(range(len(pairs)), [conv["sp"][p] for p in pairs], mk, label=lab)
ax[1].axhline(0.95, color="grey", ls=":", label="0.95 (would-be settled)")
ax[1].set_xticks(range(len(pairs))); ax[1].set_xticklabels(pairs, rotation=30, fontsize=7)
ax[1].set_ylabel("consecutive-setting Spearman"); ax[1].set_ylim(top=1.0)
ax[1].set_title("rank agreement still < 1.0 → rankings still shifting"); ax[1].legend(fontsize=8)
fig.suptitle("§1 KernelSHAP convergence over the 50-region grid (each model = its own convergence indicator)", y=1.02)
fig.tight_layout(); fig.savefig(FIG_DIR / "sleep_edf_00_ks_convergence.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved:", (FIG_DIR / "sleep_edf_00_ks_convergence.png").name)

**Conclusion.** For both models, CMI rises monotonically through n=16000 and the consecutive-setting
Spearman stays below ~0.95 — the ranking has not settled and CMI has not plateaued. **KernelSHAP does not
converge over the 50-region grid within tractable compute**, so `n=8000` was retained: doubling to 16000 does
not change the picture and roughly doubles per-sample cost (see the ms/sample columns).

## §3. CMI/PES stability vs evaluation-sample count `N`

Fix `n_samples = 8000` and ask: **how much of §1's CMI movement was evaluation noise, and is `N = 500`
enough?** Compute a per-sample **DDS pool** once over 500 stratified samples (100/class), then **bootstrap**
CMI and PES from resampled subsets of size 25 / 50 / 100 / 250 / 500 (many resamples each). Because CMI/PES
are cheap functions of the per-sample DDS, the bootstrap is instant once the pool exists.

Two answers at once:
1. **Evaluation noise** — the spread of CMI at small `N` shows how noisy the §1 (N=10) CMI values were.
2. **N = 500 adequacy** — if CMI/PES have tightened to a narrow band by `N = 500`, the ladder can commit to
   it; if still wide, `N` must grow.

**PES watch-thread:** at N=10 PES pinned at 1.00 (all 10 DDS positive), which only means "no negative DDS in
10 draws" — it cannot distinguish 1.0 from ~0.9. Larger `N` reveals whether this **nonlinear** CNN genuinely
sustains PES ≈ 1.0 or drops below Model 1's linear-artifact PES = 1.0.

> ⏱ ~10–15 min on MPS for the 500-sample DDS pool (the bootstrap itself is instant).

In [ ]:
# 1) Per-sample DDS pool at n_samples=8000 over 500 stratified samples (100/class), Model 2 seed 0.
POOL_PER_CLASS = 100                      # 500 total
pool_idx  = stratified_idx(y_test, POOL_PER_CLASS, seed=SAMPLE_SEED)
pool_sigs = [X_test[i].astype(float) for i in pool_idx]
print(f"pool: {len(pool_sigs)} samples ({POOL_PER_CLASS}/class). Computing per-sample DDS at n_samples=8000 ...")
t0 = time.perf_counter(); dds_pool = np.empty(len(pool_sigs))
for i, s in enumerate(pool_sigs):
    a = kernel_shap(pp2, s, grid, "zero", n_samples=8000, perturbations_per_eval=KS_PE)
    c = perturbation_curves(pp2, s, grid, a, method="zero")
    dds_pool[i] = decaying_degradation_score(c["MoRF"], c["LeRF"])
    if (i+1) % 50 == 0: print(f"  {i+1}/{len(pool_sigs)}  ({(time.perf_counter()-t0)/(i+1):.2f} s/sample)", flush=True)
np.save(FIG_DIR.parent / "metrics" / "methodology_dds_pool_model2_n8000.npy", dds_pool)
print(f"done in {(time.perf_counter()-t0)/60:.1f} min. DDS pool: mean={dds_pool.mean():.3f}, "
      f"frac positive={np.mean(dds_pool>0):.3f}")

In [ ]:
# 2) Bootstrap CMI and PES at each N from the DDS pool (with replacement, 300 resamples each).
rng = np.random.RandomState(SAMPLE_SEED)
SIZES = [25, 50, 100, 250, 500]; K = 300
boot = {}
for N in SIZES:
    cmis = np.empty(K); pess = np.empty(K)
    for k in range(K):
        d = dds_pool[rng.randint(0, len(dds_pool), N)]     # bootstrap resample of size N
        pv = pes_of(d); cmis[k] = CMI_of(float(np.mean(d)), pv); pess[k] = pv
    boot[N] = {"cmi": cmis, "pes": pess}

print(f"{'N':>5}{'CMI mean':>10}{'CMI std':>9}{'CMI 2.5-97.5%':>20}{'PES mean':>10}{'PES min':>9}")
for N in SIZES:
    c = boot[N]["cmi"]; p = boot[N]["pes"]
    print(f"{N:>5}{c.mean():>10.3f}{c.std():>9.3f}{f'[{np.percentile(c,2.5):.3f}, {np.percentile(c,97.5):.3f}]':>20}"
          f"{p.mean():>10.3f}{p.min():>9.3f}")
print(f"\nInterpretation: compare the CMI spread at N=25/50 (≈ the §1 regime) with N=500. "
      f"If the N=500 band is tight, 500 is adequate; watch whether PES stays ~1.0 or drops.")

In [ ]:
# Plot bootstrap spread of CMI and PES vs N.
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for j, key, ttl in [(0, "cmi", "CMI"), (1, "pes", "PES")]:
    data = [boot[N][key] for N in SIZES]
    ax[j].boxplot(data, tick_labels=[str(N) for N in SIZES], showfliers=False)
    ax[j].set_xlabel("evaluation sample count N"); ax[j].set_ylabel(f"{ttl} (bootstrap)"); ax[j].set_title(f"{ttl} stability vs N")
ax[1].axhline(1.0, color="grey", ls=":", label="PES=1.0 (linear artifact / ceiling)"); ax[1].legend(fontsize=8)
fig.suptitle("§3 CMI/PES stability vs evaluation-sample count (n_samples=8000, Model 2 seed 0)", y=1.02)
fig.tight_layout(); fig.savefig(FIG_DIR / "sleep_edf_00_cmi_stability_vs_N.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved:", (FIG_DIR / "sleep_edf_00_cmi_stability_vs_N.png").name)

## §4. Class-tracking — predicted vs true class

The sweep stratified by **true** class, but KernelSHAP and the deletion curves both attribute toward the
**predicted** class (`target_class=None` → `argmax`). On misclassified samples these differ — and with Model
2's N1 recall ≈ 0.45, N1 samples are especially likely to be misclassified. The cell reports how many of the
10 sweep samples are misclassified and which classes.

**Ladder-wide rule (CONFIRMED): attribute toward the PREDICTED class.** This is the thesis's central
**faithfulness-vs-plausibility** distinction as a concrete implementation choice:
- **CMI measures whether an explanation reflects what the model ACTUALLY COMPUTED** (faithfulness). On a
  misclassified sample the model made a decision — the *wrong* one — and the explanation must account for
  **that** decision. The deletion curves track how the **predicted-class** probability moves, which is only
  coherent for the class the model actually output.
- Attributing toward the **true** class would score an explanation of a decision the model **never made** —
  a **plausibility** question ("did the model look at the physiologically right thing?"), *not* a
  faithfulness one ("does the explanation reflect the computation?"). The two must not be conflated.
- **Not hypothetical:** 3 of the 10 stratified sweep samples are misclassified (W→N1, N1→N2, REM→N2), so
  predicted≠true affects a material fraction of the evaluation set — disproportionately **N1** (recall ≈
  0.45), the hardest minority stage.
- The reference CMI framework (Šimić et al. 2025) and the harness default both track the predicted class,
  as do concentration/oracle — so predicted keeps the whole pipeline coherent. True-class attribution stays
  available as a **separate** plausibility analysis, but the headline CMI is predicted-class, **identical
  across the ladder**.

In [ ]:
probs = pp2(np.array(SWEEP_SIGS)); pred = probs.argmax(1); true = y_test[SWEEP_IDX]
print(f"{'i':>2}{'true':>6}{'pred':>6}{'match':>7}")
mis = 0
for i in range(len(SWEEP_SIGS)):
    ok = true[i] == pred[i]; mis += (not ok)
    print(f"{i:>2}{CLASS_NAMES[true[i]]:>6}{CLASS_NAMES[pred[i]]:>6}{'y' if ok else 'NO':>7}")
byc = {}
for i in range(len(SWEEP_SIGS)):
    if true[i] != pred[i]: byc.setdefault(CLASS_NAMES[true[i]], []).append(CLASS_NAMES[pred[i]])
print(f"\nmisclassified: {mis}/{len(SWEEP_SIGS)}  | mismatches by true class: {byc}")
print("Rule adopted (see markdown): attribute toward the PREDICTED class, ladder-wide.")

## §5. Perturbation method — `zero` vs `laplace`

On per-epoch z-scored data `sample_mean ≈ zero`, so the real choice is `zero` vs `laplace`. **Primary
metric = `zero`**, for four reasons (leading with the decisive one):

1. **Stage-dependence (decisive).** `laplace` is a curvature/edge operator (discrete Laplacian `[1,-2,1]`):
   it **preserves sharp edges** (sleep spindles, K-complexes) and **flattens smooth stretches** (slow waves).
   So its "hiding strength" **varies by sleep stage** — hiding a spindle region barely changes the signal,
   hiding a slow-wave region behaves like zero. On a 5-class problem that **biases per-class faithfulness**
   by stage. `zero` removes every region's content **uniformly**.
2. **Pipeline consistency / non-circularity.** The oracle (`FeatureAblation`) and concentration both use
   `zero`; using `zero` for attribution + deletion keeps the whole faithfulness pipeline on **one** baseline.
3. **Interpretability.** On z-scored EEG, `zero` = the sample mean = "remove this region's deviation from
   baseline" — the cleanest ablation.
4. **Comparability** with the validated Model 1 machinery and ECG200, both of which used `zero` as primary.

`laplace` is retained as the **pre-registered single-model robustness check** (run on one model later, not
across the grid) — its different artefact profile is exactly what a robustness check should probe. The cell
below illustrates the stage-dependence on one spindle-bearing (N2) and one slow-wave (N3) epoch.

In [ ]:
from harness.xai.perturbation import zero_background, laplace_background
# One N2 (spindle) and one N3 (slow-wave) epoch; show a middle region hidden by zero vs laplace.
def example(stage):
    i = np.where(y_test == CLASS_NAMES.index(stage))[0][0]; return X_test[i].astype(float)
r = 25; a, b = grid.bounds[r]                       # a middle region
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for row, stage in enumerate(["N2", "N3"]):
    s = example(stage); zb = zero_background(s); lb = laplace_background(s)
    seg = slice(max(0, a-120), min(len(s), b+120))
    ax[row].plot(np.arange(len(s))[seg], s[seg], color="k", lw=.8, label="original")
    z = s.copy(); z[a:b] = zb[a:b]; l = s.copy(); l[a:b] = lb[a:b]
    ax[row].plot(np.arange(len(s))[seg], z[seg], color="tab:blue", lw=1.4, label="region hidden: zero")
    ax[row].plot(np.arange(len(s))[seg], l[seg], color="tab:red", lw=1.4, label="region hidden: laplace")
    ax[row].axvspan(a, b, color="grey", alpha=.15)
    ax[row].set_ylabel(f"{stage} epoch"); ax[row].legend(fontsize=8, loc="upper right")
    resid = np.abs(s[a:b] - lb[a:b]).mean()
    ax[row].set_title(f"{stage}: mean |original − laplace| in region = {resid:.3f}  "
                      f"(large ⇒ laplace barely hides high-freq content)", fontsize=9)
ax[1].set_xlabel("timestep")
fig.suptitle("§5 zero hides uniformly; laplace's hiding strength depends on local curvature (stage-dependent)", y=1.0)
fig.tight_layout(); fig.savefig(FIG_DIR / "sleep_edf_00_pm_zero_vs_laplace.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved:", (FIG_DIR / "sleep_edf_00_pm_zero_vs_laplace.png").name)

## §6. Device selection

Per-method device choice was **measured**, not assumed (Model 2 seed 0, 50-region grid). Attributions agree
across devices to **max|Δ| ≈ 1e-7–2e-6** (negligible), so device is purely a speed choice; and coalition
batching changes KernelSHAP attributions by only **max|Δ| = 8e-8** (`pe=1` vs `pe=200`), so batching is a
compute-layout choice, not a scientific one.

| method | CPU | MPS | chosen | why |
|---|---|---|---|---|
| **FeatureAblation** | **44 ms/sample** | 451 ms | **CPU** | 50 tiny batch-1 forwards; MPS transfer overhead dominates |
| **KernelSHAP (unbatched, pe=1)** | 2.7 s (n=2000) | 3.9 s | CPU | many tiny batch-1 forwards |
| **KernelSHAP (batched, pe=200)** | 35.9 s (n=2000) ✗ | **0.37 s (n=2000) / 1.34 s (n=8000)** | **MPS-batched** | one batched forward per 200 coalitions — MPS parallelises; CPU can't |
| **Integrated Gradients** | 1.27 s | **0.71 s** | **MPS** | one batched 50-step forward+backward |

**The KernelSHAP device choice is contingent on batching:** *unbatched* it is faster on CPU; *batched* it
must run **MPS** (batching is a 13× loss on CPU but a 13× win on MPS). This is why `perturbations_per_eval`
was added to the harness (default 1, equivalence-gated at 8e-8). The cell re-confirms the headline numbers
live on a couple of samples.

> ⏱ ~30 s.

In [12]:
from harness.xai.feature_ablation import feature_ablation
from harness.xai.integrated_gradients import integrated_gradients
demo = SWEEP_SIGS[:2]
def tmethod(fn):
    t0 = time.perf_counter(); [fn(s) for s in demo]; return (time.perf_counter()-t0)/len(demo)*1000

m_cpu = load_model2(0, "cpu"); pp_cpu = torch_predict_proba(m_cpu, device="cpu")
print("FeatureAblation  CPU: %5.0f ms/sample" % tmethod(lambda s: feature_ablation(pp_cpu, s, grid, "zero")))
print("KernelSHAP pe=1  CPU: %5.0f ms/sample (n=2000)" % tmethod(lambda s: kernel_shap(pp_cpu, s, grid, "zero", n_samples=2000, perturbations_per_eval=1)))
if DEVICE == "mps":
    m_mps = load_model2(0, "mps"); pp_mps = torch_predict_proba(m_mps, device="mps")
    print("KernelSHAP pe=200 MPS: %5.0f ms/sample (n=2000)" % tmethod(lambda s: kernel_shap(pp_mps, s, grid, "zero", n_samples=2000, perturbations_per_eval=200)))
    print("IntegratedGradients MPS: %5.0f ms/sample" % tmethod(lambda s: integrated_gradients(m_mps, s, grid, "zero")))
# cross-device attribution agreement (FA) + batching equivalence (KS) as evidence
a_cpu = feature_ablation(pp_cpu, demo[0], grid, "zero")
if DEVICE == "mps":
    a_mps = feature_ablation(pp_mps, demo[0], grid, "zero")
    print(f"\nFA cross-device max|Δ| = {np.max(np.abs(a_cpu-a_mps)):.2e}")
k1 = kernel_shap(pp_cpu, demo[0], grid, "zero", n_samples=2000, perturbations_per_eval=1)
k2 = kernel_shap(pp_cpu, demo[0], grid, "zero", n_samples=2000, perturbations_per_eval=200)
print(f"KS batching equivalence pe=1 vs pe=200 max|Δ| = {np.max(np.abs(k1-k2)):.2e}")

FeatureAblation  CPU:    55 ms/sample
KernelSHAP pe=1  CPU:  1122 ms/sample (n=2000)
KernelSHAP pe=200 MPS:   416 ms/sample (n=2000)
IntegratedGradients MPS:    38 ms/sample

FA cross-device max|Δ| = 1.19e-07
KS batching equivalence pe=1 vs pe=200 max|Δ| = 8.01e-08


### §6.1 Same-device repeat-determinism  (▶ short real run, ~1 min)

§6 shows attributions **agree across devices** to ~1e-7 — but that is not literally the claim that *the same
run reproduces*. This checks that directly: train a shallow CNN **twice** with identical seed and data on the
same device and compare all weights. Bit-identical weights ⇒ the pipeline is deterministic on this device.

> ▶ **Real run** (short): two 2-epoch trainings on a 400/100 split, on `DEVICE`. ~1 min. `train_cnn` seeds both
> `set_seed(seed)` and the DataLoader generator, so given the same seed/data the two runs should be identical.

In [13]:
# §6.1 — same-device repeat-determinism: train twice (identical seed/data), assert bit-identical weights.
from harness.models.cnn import set_seed, train_cnn
Xtr_det, ytr_det = load_sleep_edf("train", verbose=False)
_di = stratified_idx(ytr_det, 100, SAMPLE_SEED)                 # 500 stratified samples, fixed
np.random.RandomState(SAMPLE_SEED).shuffle(_di)                 # shuffle so BOTH train & val carry all 5 classes
_Xd, _yd = Xtr_det[_di], ytr_det[_di]; _cut = 400

def _train_once():
    set_seed(0)
    m = build_cnn("shallow", n_classes=cfg.N_CLASSES, in_channels=cfg.IN_CHANNELS, kernel_size=cfg.CNN_KERNEL_SIZE)
    train_cnn(m, _Xd[:_cut], _yd[:_cut], _Xd[_cut:], _yd[_cut:], max_epochs=2, patience=2, seed=0, device=DEVICE)
    return {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}

print(f"Training a shallow CNN twice (identical seed/data) on {DEVICE} ...", flush=True)
_sd1 = _train_once(); _sd2 = _train_once()
_maxdiff = max(float((_sd1[k] - _sd2[k]).abs().max()) for k in _sd1)
print(f"\nsame-device ({DEVICE}) repeat-training  max|delta| over ALL params = {_maxdiff:.3e}")
assert _maxdiff == 0.0, f"NON-DETERMINISTIC on {DEVICE}: max|delta| = {_maxdiff:.2e} (expected exactly 0)"
print(">>> BIT-IDENTICAL: same-device training is deterministic. This is the literal same-device claim;")
print("    the §6 cross-device max|delta| ~1e-7 is a separate (weaker) statement about CPU-vs-MPS agreement.")

Training a shallow CNN twice (identical seed/data) on mps ...

same-device (mps) repeat-training  max|delta| over ALL params = 0.000e+00
>>> BIT-IDENTICAL: same-device training is deterministic. This is the literal same-device claim;
    the §6 cross-device max|delta| ~1e-7 is a separate (weaker) statement about CPU-vs-MPS agreement.


## Summary of decisions (recorded in DECISIONS_LOG.md)

- **`n_samples = 8000`** for KernelSHAP — a decision *and a documented limitation* (§1/§2). KS over 50
  regions does not fully converge within tractable compute; absolute CMI is biased low. Absolute CMI values
  are comparable **within this study only**, not against published CMI figures.
- **Evaluation `N`** — set from §3 (bootstrap stability of CMI/PES).
- **Target class = predicted** (§4), identical across the ladder.
- **Perturbation method = `zero`** primary; `laplace` = pre-registered single-model robustness check (§5).
- **Device** — FeatureAblation CPU, KernelSHAP MPS-batched (`perturbations_per_eval`), Integrated Gradients
  MPS (§6); choices verified not to affect results (≤ 2e-6).